In [1]:
# state: n_guesses, lower_bound, upper_bound, last_guess
# action: guess a number 'n' in the given range
# reward: correct guess +10, wrong guess -1
# termination: correct guess is made

# Q(s, a) -> How good is action 'a' in state 's'

In [2]:
import random
import torch
import matplotlib.pyplot as plt

is_ipython = 'inline' in plt.get_backend()
if is_ipython:
    from IPython import display

seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

In [3]:
import torch
from itertools import count
from collections import namedtuple
import random
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device("cuda")

TAU = 0.005 # Update rate of target network
GAMMA = 0.99 # Discount factor
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
LR = 3e-4
BS = 128



steps_done = 0

def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1.0 * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        # exploit
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        # explore
        return torch.tensor([[random.randint(0, 99)]], device=device, dtype=torch.long)

def step(state, action, goal_number):
    min_val = int(state[0, 1].item())
    max_val = int(state[0, 2].item())
    step_count = int(state[0, 0].item())

    actual_action = min(max(min_val, action), max_val)

    if action == goal_number:
        return None, 10, True

    if actual_action > goal_number:
        new_max = actual_action - 1
        new_min = min_val
    else:
        new_min = actual_action + 1
        new_max = max_val

    if new_min > new_max:
        return None, -10, True

    direction = 1 if actual_action > goal_number else -1
    return [step_count + 1, new_min, new_max, actual_action, direction], -1, False

def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title("Result")
    else:
        plt.clf()
        plt.title("Training...")
    plt.xlabel("Episode")
    plt.ylabel("Duration")
    plt.plot(durations_t.numpy())
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.axhline(y=7, color='green', linestyle='--', linewidth=2, label='Optimal')
    plt.legend()
    plt.pause(0.001)
    if not show_result:
        display.display(plt.gcf())
        display.clear_output(wait=True)
    else:
        display.display(plt.gcf())


Transition = namedtuple("Transition", ("state", "action", "next_state", "reward"))

class Memory:
    def __init__(self, capacity):
        self.memory = []
        self.capacity = capacity

    def push(self, *args):
        self.memory.append(Transition(*args))
        if len(self.memory) > self.capacity:
            self.memory.pop(0)
    
    def sample(self, bs):
        return random.sample(self.memory, bs)
    
    def __len__(self):
        return len(self.memory)
    
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.l1 = nn.Linear(n_observations, 128)
        self.l2 = nn.Linear(128, 128)
        self.l3 = nn.Linear(128, n_actions)
    
    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        return self.l3(x)

policy_net = DQN(5, 100).to(device)
target_net = DQN(5, 100).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = torch.optim.Adam(policy_net.parameters(), lr=LR)
memory = Memory(capacity=10000)

def optimize():
    if len(memory) < BS:
        return
    transitions = memory.sample(BS)
    batch = Transition(*zip(*transitions))

    non_final_mask = torch.tensor([s is not None for s in batch.next_state], device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = policy_net(state_batch).gather(1, action_batch)
    next_state_values = torch.zeros(BS, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    loss = F.smooth_l1_loss(state_action_values, expected_state_action_values.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()


episode_durations = []
episode_rewards = []

n_episodes = 5000
for episode in range(1, n_episodes + 1):
    goal_number = random.randint(1, 100)
    state = torch.tensor([0, 1, 100, 0, 0], device=device, dtype=torch.float32).unsqueeze(0)
    for t in count():
        action = select_action(state)
        observation, reward, done = step(state, action, goal_number)
        episode_rewards.append(reward)
        reward = torch.tensor([reward], device=device)

        if done:
            next_state = None
        else:
            next_state = torch.tensor(observation, device=device, dtype=torch.float32).unsqueeze(0)
        
        memory.push(state, action, next_state, reward)
        state = next_state

        optimize()

        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            # plot_durations()
            break;
    
    if episode % 1000 == 0:
        recent_durations = episode_durations[-50:]
        recent_rewards = episode_rewards[-50:]
        
        avg_duration = sum(recent_durations) / len(recent_durations)
        avg_reward = sum(recent_rewards) / len(recent_rewards)
        
        print(f"Episode {episode}: Avg steps={avg_duration:.1f}, Avg reward={avg_reward:.1f}")
        
        # Check if converged
        if episode >= 500:
            last_100 = episode_durations[-100:]
            if max(last_100) - min(last_100) < 3:  # Very stable
                print(f"Converged at episode {episode}!")
                # Can stop or continue for fine-tuning


# plot_durations(show_result=True)
plt.ioff()
plt.show()

Episode 1000: Avg steps=8.8, Avg reward=-1.9
Episode 2000: Avg steps=7.6, Avg reward=-1.0
Episode 3000: Avg steps=7.6, Avg reward=-1.9
Episode 4000: Avg steps=7.6, Avg reward=-2.0
Episode 5000: Avg steps=7.3, Avg reward=-1.9


In [4]:
num_test_episodes = 5
for episode in range(num_test_episodes):
    goal_number = random.randint(1, 100)
    state = torch.tensor([0, 1, 100, 0, 0], device=device, dtype=torch.float32).unsqueeze(0)
    actions_taken = []
    states_visited = []
    
    print(f"\n{'='*60}")
    print(f"Episode {episode + 1}")
    print(f"{'='*60}")
    print(f"Starting state: step={int(state[0,0])}, range=[{int(state[0,1])}, {int(state[0,2])}], last_action={int(state[0,3])}")
    
    for t in count():
        # Get action (in exploitation mode for testing)
        with torch.no_grad():
            q_values = policy_net(state)
            action = q_values.max(1).indices.item()
        
        # Extract current range for mapping
        min_val = int(state[0, 1].item())
        max_val = int(state[0, 2].item())
        actual_action = min(max(min_val, action), max_val)
        
        actions_taken.append(actual_action)
        states_visited.append([int(state[0,0]), int(state[0,1]), int(state[0,2]), int(state[0,3])])
        
        print(f"  Step {t+1}: Choose {actual_action} from range [{min_val}, {max_val}] (DQN output: {action})")
        
        observation, reward, done = step(state, action, goal_number)
        
        if not done:
            state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
            print(f"          -> New range: [{int(state[0,1])}, {int(state[0,2])}], reward={reward}")
        else:
            print(f"          -> GOAL REACHED! Reward={reward}")
            print(f"\n  Total steps: {t + 1}")
            print(f"  Action sequence: {actions_taken}")
            break



Episode 1
Starting state: step=0, range=[1, 100], last_action=0
  Step 1: Choose 40 from range [1, 100] (DQN output: 40)
          -> New range: [1, 39], reward=-1
  Step 2: Choose 19 from range [1, 39] (DQN output: 19)
          -> New range: [20, 39], reward=-1
  Step 3: Choose 34 from range [20, 39] (DQN output: 34)
          -> New range: [20, 33], reward=-1
  Step 4: Choose 31 from range [20, 33] (DQN output: 31)
          -> New range: [20, 30], reward=-1
  Step 5: Choose 28 from range [20, 30] (DQN output: 28)
          -> New range: [20, 27], reward=-1
  Step 6: Choose 27 from range [20, 27] (DQN output: 28)
          -> GOAL REACHED! Reward=-10

  Total steps: 6
  Action sequence: [40, 19, 34, 31, 28, 27]

Episode 2
Starting state: step=0, range=[1, 100], last_action=0
  Step 1: Choose 40 from range [1, 100] (DQN output: 40)
          -> New range: [1, 39], reward=-1
  Step 2: Choose 19 from range [1, 39] (DQN output: 19)
          -> New range: [1, 18], reward=-1
  Step 3: C